# 03. SARIMA Training

Training SARIMA models for Onion, Potato, and Tomato using statsmodels.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import joblib
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

df = pd.read_csv("../backend/data/processed/engineered_features.csv")
df["date"] = pd.to_datetime(df["date"])

os.makedirs("../backend/models/saved", exist_ok=True)

def train_sarima(commodity):
    data = df[df["commodity"] == commodity].set_index("date")["price"]
    train, test = data.iloc[:-30], data.iloc[-30:]
    
    print(f"Training SARIMA for {commodity}...")
    # Using simplified order for faster execution in MVP
    model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12))
    result = model.fit(disp=False)
    
    # Forecast
    forecast = result.get_forecast(steps=30)
    pred = forecast.predicted_mean
    
    mae = mean_absolute_error(test, pred)
    mape = mean_absolute_percentage_error(test, pred)
    print(f"{commodity} SARIMA - MAE: {mae:.2f}, MAPE: {mape:.4f}")
    
    # Save model
    joblib.dump(result, f"../backend/models/saved/sarima_{commodity}.pkl")
    
    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(train.index[-60:], train.iloc[-60:], label="Train")
    plt.plot(test.index, test, label="Actual")
    plt.plot(test.index, pred, label="Forecast")
    plt.fill_between(test.index, forecast.conf_int().iloc[:, 0], forecast.conf_int().iloc[:, 1], color='k', alpha=.15)
    plt.title(f"SARIMA Forecast - {commodity.capitalize()}")
    plt.legend()
    plt.savefig(f"03_sarima_{commodity}.png")
    plt.close()

for c in ["onion", "potato", "tomato"]:
    train_sarima(c)